In [ ]:
library(ggplot2)
library(tidyverse)
library(gridExtra)
library(GGally)
library(plotly)
library(corrplot)
library(reshape2)
library(FactoMineR) 
library(factoextra)
library(glmnet) 
library(ggfortify)
library(pROC)
library(ROCR)
library(GGally)
library(dplyr)

In [ ]:
data<-read.csv("new_games.csv",header = TRUE)

In [ ]:
summary(data)

# Conversion des variables lues en tant que quantitatives en qualitatives

Il me semble que les variables ci-dessous sont plutôt des variables qualitatives donc je les ai converti.

In [ ]:

data[,"Estimated.owners"]=as.factor(data[,"Estimated.owners"])
data[,"Required.age"]=as.factor(data[,"Required.age"])
data[,"Windows"]=as.factor(data[,"Windows"])
data[,"Mac"]=as.factor(data[,"Mac"])
data[,"Linux"]=as.factor(data[,"Linux"])


summary(data)

       

#  Suppresion de variables inutiles à l'exploration des données

J'ai choisi de supprimer plusieurs variable qui à mon avis ne seraient pas pertinente lors de l'analyse de certaine donnée. J'ai aussi supprimé la variable Movies car elle ne contennait que des NA. score rank ne contient que 40 valeurs.

In [ ]:
data1 <- data %>% select(-c(AppID, About.the.game, Reviews,Movies,Header.image,Support.email,Support.url,Notes,Metacritic.url,Screenshots,Score.rank,Website))

In [ ]:
head(data1)

# Analyse de la qualités des individus (les jeux vidéos): filtrage

Avant de commencer notre analyse, nous allons faire un premier filtrage sur les individus. En effet, nous cherchons à supprimer pour les variables suivantes les cases vides, les cases contenant des NAN et pour Supported.language les cases ne contenant pas de valeurs soit $[]$:Publishers,Developers,Name,Supported.languages,Genres. Nous avons effectué ce filtage car nous avons jugé inutilisable les individus(les jeux) qui au minimun n'avaient aucunes données pour ces variables là.

In [ ]:

for (col in colnames(data1))
{

    print((paste(col,sum(is.na(data1[,col]) | data1[,col] == "" |data1[,col] == "[]" ))))
    
}

In [ ]:
data_filtered <- data1 %>%
  filter(
    !is.na(Name) & Name != "",
    !is.na(Developers) & Developers != "",
    !is.na(Publishers) & Publishers != "",
    !is.na(Categories) & Categories != "",
    !is.na(Genres) & Genres != "",
    Supported.languages != "[]" & !is.na(Supported.languages)
  )

In [ ]:
for (col in colnames(data_filtered)) {
    # On calcule le nombre de vides/NA/[]
    nb_vides <- sum(is.na(data_filtered[[col]]) | 
                    data_filtered[[col]] == "" | 
                    data_filtered[[col]] == "[]", 
                    na.rm = TRUE)
    
    # On affiche le résultat de manière lisible
    print(paste(col, ":", nb_vides))
}

In [ ]:
nbr_suppr <- nrow(data1) - nrow(data_filtered)

print(paste("Nous avons supprimé", nbr_suppr, "individus."))

## Traitement des doublons (jeux qui ont le même nom)

In [ ]:
doublons <- data_filtered %>%
  group_by(Name) %>%
  filter(n() > 1) %>%
  arrange(Name)
head(doublons, 20)

total_doublons <- nrow(doublons)

noms_problematiques <- n_distinct(doublons$Name)

print(paste("Il y a", total_doublons, "lignes qui concernent", noms_problematiques, "noms de jeux en double."))

Après analyse, nous avons identifié la présence de doublons homonymes (jeux partageant le même nom mais possédant des caractéristiques distinctes). Faute de critère d'exclusion garantissant la non-suppression de versions légitimes (remakes, éditions régionales, ou suites éponymes), nous avons fait le choix de garder le jeux de données tel quel.

# Etude sur toute les données sans filtrage approfondies

# Variables quantitatives

On va d'abord afficher toutes les variables restantes pour voir si pour pouvoir les étudier il faut réaliser des transformations

In [ ]:

for (col in colnames(data_filtered)) {
  # On vérifie si la colonne est est celle d'une variable quantitatives
  if(is.numeric(data_filtered[[col]])) {
    
    p <- ggplot(data_filtered, aes(x = .data[[col]])) +
      geom_histogram(aes(y = after_stat(density)), alpha = 0.6, fill = "gray",bins=100) +
      theme_minimal() +
      labs(title = paste("Distribution de :", col))
    
    print(p)
  }
}

Puisque nous n'avons pas encore filtré les variables et que pour beaucoup de jeux les valeurs sont à 0 on va faire une transformation en log.

In [ ]:
data_log=data_filtered
data_log[,"LPeak.CCU"]=log(1+data_log[,"Peak.CCU"])
data_log[,"LPrice"]=log(1+data_log[,"Price"])
data_log[,"LDiscount"]=log(1+data_log[,"Discount"])
data_log[,"LDLC.count"]=log(1+data_log[,"DLC.count"])
data_log[,"LMetacritic.score"]=log(1+data_log[,"Metacritic.score"])
data_log[,"LUser.score"]=log(1+data_log[,"User.score"])
data_log[,"LPositive"]=log(1+data_log[,"Positive"])
data_log[,"LNegative"]=log(1+data_log[,"Negative"])
data_log[,"LAchievements"]=log(1+data_log[,"Achievements"])
data_log[,"LRecommendations"]=log(1+data_log[,"Recommendations"])
data_log[,"LAverage.playtime.forever"]=log(1+data_log[,"Average.playtime.forever"])
data_log[,"LAverage.playtime.two.weeks"]=log(1+data_log[,"Average.playtime.two.weeks"])
data_log[,"LMedian.playtime.forever"]=log(1+data_log[,"Median.playtime.forever"])
data_log[,"lMedian.playtime.two.weeks"]=log(1+data_log[,"Median.playtime.two.weeks"])


In [ ]:
data_log <- data_log %>% select(-c(Peak.CCU,Price,Discount,DLC.count,Metacritic.score,User.score,Positive,Negative,Achievements,Recommendations,Average.playtime.forever,Average.playtime.two.weeks,Median.playtime.forever,Median.playtime.two.weeks))

In [ ]:
for (col in colnames(data_log)) {
  # On vérifie si la colonne est est celle d'une variable quantitatives
  if(is.numeric(data_log[[col]])) {
    
    p <- ggplot(data_log, aes(x = .data[[col]])) +
      geom_histogram(aes(y = after_stat(density)), alpha = 0.6, fill = "gray",bins=100) +
      theme_minimal() +
      labs(title = paste("Distribution de :", col))
    
    print(p)
  }
}

**A compléter** avec ce qu'on avait vu sur le notebook de jocelin.

Donc même en essayant de transformer les données, on se rends compte que les variables ne sont pas lisibles tant que l'on ne filtre pas les données: c'est à dire de supprimer les individus qui ont des valeurs nulles.

# Variables qualitatives

In [ ]:

# On définit la liste de TES variables (car is.factor risque de rater le coche)
colonnes_qualitatives <- c("Required.age", "Windows", "Mac", "Linux", "Estimated.owners")

for (col in colonnes_qualitatives) {
    # On utilise geom_bar() car il compte les lignes automatiquement
    p <- ggplot(data_filtered, aes(x = .data[[col]])) +
      geom_bar(fill = "steelblue", color = "white") +
      theme_minimal() +
      labs(
        title = paste("Répartition de la variable :", col),
        x = col,
        y = "Nombre de jeux"
      )
    
    # Gestion spécifique pour la lisibilité des owners
    if (col == "Estimated.owners") {
      p <- p + coord_flip()
    }
    
    print(p)

}

Pour les variables qualitatives, nous pouvons conclure que la quasi-totalité des jeux sont disponible sur Windows et seulement qu'une petite partie sur Windows et Linux.
De plus, il n'y a que très peu de jeux qui qui ont beaucoup d'utilisateurs, la plupart ont entre 0 et 20 000 déteneurs du jeu.
Enfin, on remarque que très peu de jeux ont des restriction d'âge

 **Etude qu'on peut faire**: 
 - quels sont les individus(jeux) qui sont présent sur linux et Mac,peut etre les jeux qui sont les plus connus
 - est- ce que les jeux qui ont des restriction d'age marche mieux ou moins bien que les jeux sans?
 - ...

# Filtrage

## Etude variables par variables: gestion et suppression outliers

### Etude sur la variable supported.language

In [ ]:
head(data_filtered[,"Supported.languages"])

Plutôt que de garder la variable supported.languages qui ne contient que des chaines de caractère. On peut plutôt étudier le nombre de jeu supporté par chaque langue (première cellule) ou par jeux le nombre de langue supporté et est ce que ça à une influence sur la note du jeu ou autre chose (deuxième cellule).

In [ ]:
stats_langues <- data_filtered %>%
  # 1. On nettoie les symboles parasites [ ] ' "
  mutate(clean_txt = str_remove_all(Supported.languages, "\\[|\\]|'|\"")) %>%
  # 2. On "éclate" les listes : si un jeu a 3 langues, ça crée 3 lignes
  separate_rows(clean_txt, sep = ",\\s*") %>%
  # 3. On compte combien de fois chaque langue apparaît
  count(clean_txt, sort = TRUE) %>%
  # 4. On garde les 50 premières (en enlevant les vides)
  filter(clean_txt != "" & !is.na(clean_txt)) %>%
  head(40)


ggplot(stats_langues, aes(x = reorder(clean_txt, n), y = n)) +
  geom_col(fill = "steelblue") +
  coord_flip() + # Pour mettre les noms à l'horizontale 
  labs(
    title = "Top 40 des langues les plus supportées sur Steam",
    x = "Langue",
    y = "Nombre de jeux"
  ) +
  theme_minimal()

In [ ]:
data_filtered <- data_filtered|>
  mutate(nb_language_supported = case_when(          
    TRUE ~ str_count(Supported.languages, ",") + 1  
  ))
head(data_filtered$nb_language_supported)

### Etude des variables developers et publishers

Je propose de faire de même pour les variables developers et publishers: c'est à dire de regarder si la note du jeu a un lien avec sont studio de développement et son publieur. J'ai converti les chaines de caractères en minuscule et envelevé les points car dans un premier temps j'ai vu que certains chaines de caractère étaient la même mais juste en majuscule .

In [ ]:

# 1. On définit la liste des suffixes à nettoyer (le "bruit")
# Tu peux en ajouter d'autres ici si tu en vois de nouveaux dans tes données
bruit_juridique <- c("LTD", "INC", "LLC", "CORP", "CO", "SA", "SAS", "GMBH", "KK", "SL", "AS")

# On crée le "pattern" de recherche (Regex) : 
# Il cherche ces mots uniquement à la fin de la ligne (d'où le $)
pattern_bruit <- paste0("\\s*\\b(", paste(bruit_juridique, collapse = "|"), ")\\b[. ]*$")

# 2. Traitement du dataset
stat_dev_clean <- data_filtered |>
  # Nettoyage des crochets et guillemets de la liste brute
  mutate(Developers = str_remove_all(Developers, "['\\[\\]]")) |>
  
  # On sépare les jeux en multi-développeurs (une ligne par studio)
  separate_rows(Developers, sep = ",\\s*") |>
  
  # Nettoyage et standardisation
  mutate(
    dev_final = Developers |>
      str_to_upper() |>              # Tout en MAJ pour fusionner (ltd = LTD)
      str_remove_all("\\.") |>       # On enlève les points (INC. -> INC)
      str_remove_all(pattern_bruit)|> # On enlève les suffixes définis plus haut
      str_trim()                     # On enlève les espaces restants
  ) |>
  
  # On évite de compter deux fois le même dev sur un même jeu
  distinct() |>
  
  # On compte et on filtre
  count(dev_final, sort = TRUE) |>
  filter(dev_final != "" & !is.na(dev_final)) |>
  head(40)
  

# 3. Création du graphique
ggplot(stat_dev_clean, aes(x = reorder(dev_final, n), y = n)) +
  geom_col(fill = "steelblue") +
  coord_flip() +
  labs(
    title = "Top 40 des développeurs les plus actifs sur Steam",
    subtitle = "Données nettoyées (suffixes juridiques harmonisés)",
    x = "Développeur",
    y = "Nombre de jeux"
  ) +
  theme_minimal()



In [ ]:

# 1. On définit la liste des suffixes à harmoniser (le "bruit")
bruit_juridique <- c("LTD", "INC", "LLC", "CORP", "CO", "SA", "SAS", "GMBH", "KK", "SL", "AS")
pattern_bruit <- paste0("\\s*\\b(", paste(bruit_juridique, collapse = "|"), ")\\b[. ]*$")

# 2. Nettoyage et préparation des données
stat_publisher <-data_filtered |>
  # Nettoyage des symboles de liste et des textes entre parenthèses (ex: (Mac))
  mutate(clean_publ = Publishers |> 
           str_remove_all("['\\[\\]]") |> 
           str_remove_all("\\(.*?\\)")) |> 
  
  # On sépare les co-éditeurs (une ligne par éditeur)
  separate_rows(clean_publ, sep = ",\\s*") |>
  
  # STANDARDISATION POUSSÉE
  mutate(
    clean_publ = clean_publ |>
      str_to_upper() |>              # Tout en MAJUSCULES pour fusionner les variantes
      str_remove_all("\\.") |>       # On enlève les points (INC. -> INC)
      str_remove_all(pattern_bruit) |> # On enlève les suffixes (LTD, LLC...)
      str_trim()                     # On nettoie les espaces restants
  ) |>
  
  # On évite de compter deux fois le même éditeur pour un même jeu (doublons internes)
  distinct() |>
  
  # On compte
  count(clean_publ, sort = TRUE) |>
  filter(clean_publ != "" & !is.na(clean_publ)) |>
  
  # On prend les 30 premiers pour le graphique
  head(30)

# 3. Visualisation moderne
ggplot(stat_publisher, aes(x = reorder(clean_publ, n), y = n)) +
  geom_col(fill = "steelblue") +
  coord_flip() + # On garde coord_flip ou on inverse x et y dans aes()
  labs(
    title = "Top 30 des éditeurs sur Steam",
    subtitle = "Données nettoyées : suffixes supprimés et collaborations séparées",
    x = "Éditeurs",
    y = "Nombre de jeux"
  ) +
  theme_minimal()


### Etude variable release.date

A voir  comment les traiter: en variable qualitative ou autre .

### Etude required.age

In [ ]:
ggplot(data_filtered, aes(x = factor(1),fill = Required.age))+ geom_bar(width = 1) +
  coord_polar("y") +
  theme_void() +
  labs(
    title = "Répartition des jeux par limite d'âge",
    fill = "Âge Requis"
  )

On remarque que les jeux qui n'ont pas de limites sont en grande majorité qui rends la lecture du pie chart compliquée. Il faudrait enlevé cette modalité pour réaliser la stat descriptive. C'est ce qui est fait dans la cellule suivante.

In [ ]:


# filtrage de 0
data_age <- data_filtered |> 
  filter(as.character(Required.age) != "0")


ggplot(data_age, aes(x = factor(1), fill = factor(Required.age))) +
  geom_bar(width = 1) +
  coord_polar("y") +
  theme_void() + 
  labs(
    title = "Répartition des restrictions d'âge (Hors 0+)",
    fill = "Âge"
  )

On remarque une majorité de restriction pour les moins de 17 ans.

# Traitement des outliers

In [ ]:

data2 <- data_filtered[data_filtered$Estimated.owners != "0 - 0", ]

dim(data2)


In [ ]:

data2$Positive=data2$Positive+1
data2$Negative=data2$Negative+1


data2[, "LPositive"] <- log(data2[, "Positive"])

data2[, "LNegative"] <- log(data2[, "Negative"])


In [ ]:
g1<-ggplot(data1,aes(x=Positive))+ geom_histogram(alpha=0.6,aes(y=after_stat(density)),bins=100)
g2<-ggplot(data1,aes(x=Negative)) + geom_histogram(alpha=0.6,aes(y=after_stat(density)),bins=100)
 grid.arrange(g1,g2,ncol=2)

In [ ]:



 lg1<-ggplot(data2,aes(x=LPositive))+geom_density(linewidth=1,col="darkolivegreen") + geom_histogram(alpha=0.6,aes(y=after_stat(density)))
 lg2<-ggplot(data2,aes(x=LNegative))+geom_density(linewidth=1,col="darkolivegreen") + geom_histogram(alpha=0.6,aes(y=after_stat(density)))  

 grid.arrange(lg1,lg2,ncol=2)

# Concaténation variable Tag et genres

On concatène les variables Tags et Genres car on s'est rendu compte que la variables Tags contenait la variable Genres la plus part du temps.

In [ ]:

data3 <- data2 %>%
  # 1. On travaille ligne par ligne
  rowwise() %>%
  
  mutate(
    # 2. On fusionne les deux colonnes
    temp_concat = paste(Tags, Genres, sep = ","),
    
    # 3. On nettoie à l'intérieur de la ligne
    Tags_Genres_Final = temp_concat %>%
      # On découpe par la virgule
      str_split(",") %>%
      # On transforme en vecteur simple
      unlist() %>%
      # On enlève les espaces inutiles
      str_trim() %>%
      # On enlève les vides, les "NA", "NaN", "null" (insensible à la casse)
      .[. != "" & !is.na(.) & !str_detect(tolower(.), "^na$|nan|null")] %>%
      # On garde les valeurs uniques (supprime les doublons entre Tags et Genres)
      unique() %>%
      # On recolle le tout
      paste(collapse = ", ")
  ) %>%
  
  # 4. On repasse en mode normal et on supprime la colonne temporaire
  ungroup() %>%
  select(-temp_concat)


In [ ]:
head(data3)

### On enleve les anciennes variables que l'on a modifié

In [ ]:
data4 <- data3 %>% select(-c(Tags,Genres,Positive,Negative))

In [ ]:
summary(data4)

## Etude variariable métacrique score

C'est une variable très importante qu'il faudra exploiter. C'est une note moyenne donnée par des journalistes, des magazines de jeux vidéo et des sites web experts. Cette note est entre 0 et 100 plus on est proche de 100 mieux c'est. Cependant, beaucoup de jeux n'ont pas de note car peux de journaux ont parlé d'eux assez pour que Steam en prenne compte dans une note, donc qu'il est une note different de 0.

In [ ]:

100-(sum(data4$Metacritic.score == 0)/length(data$Metacritic.score))*100

En effet, il y a que 21% des jeux qui ont une note superieur à 0. La question c'est est ce qu'on garde que c'est jeux là pour une étude pour prédire le score metacritic?  

### Etude temps de jeux

In [ ]:
ggplot(data4, aes(x = Average.playtime.forever)) +
  geom_histogram(fill = "skyblue", color = "white", bins = 30) +
  theme_minimal() +
  labs(title = "Distribution du temps de jeu (Échelle Logarithmique)")

ggplot(data4, aes(x = Median.playtime.forever)) +
  geom_histogram(fill = "skyblue", color = "white", bins = 30) +
  theme_minimal() +
  labs(title = "Distribution du temps de jeu (Échelle Logarithmique)")

In [ ]:
sum(data4$Average.playtime.forever==0)
sum(data4$Median.playtime.forever==0)

On se rend compte que le nombre de jeux qui on un temps de jeux égale à 0 fausse l'étude: on a qu'un seul pic à 0. Aussi, on remarque que l'on a une échelle en x de 1e+06 ce qui veut dire que l'on a aussi des jeux qui ont un temps de jeux énorme donc en plus d'enlever les 0 on va transformer en log les données.

In [ ]:

data5 <- data4 %>%
    filter(
    Average.playtime.forever > 0,         
    Median.playtime.forever > 0
 
  ) %>%
  # 2. On calcule les versions Log
  mutate(
    LAverage_playtime = log(Average.playtime.forever),
    LMedian_playtime = log(Median.playtime.forever)
  ) 

# Visualisation pour vérifier l'effet du Log
ggplot(data5, aes(x = LAverage_playtime)) +
  geom_histogram(fill = "skyblue", color = "white", bins = 30) +
  theme_minimal() +
  labs(title = "Distribution du temps de jeu (Échelle Logarithmique)")

ggplot(data5, aes(x = LMedian_playtime)) +
  geom_histogram(fill = "skyblue", color = "white", bins = 30) +
  theme_minimal() +
  labs(title = "Distribution du temps de jeu (Échelle Logarithmique)")

Ainsi, on peut se satisfaire de ces graphs.**on pourra supprimerAverage.playtime.forever, Median.playtime.forever et garder la version log**

# Etude influence estimated_owners

In [ ]:
head(data5$Estimated.owners)

Pour que ça soit plus clair je vais renommer les variables car on peut se perdre dans la notation

In [ ]:
data_rename <- data5 %>%
  mutate(Estimated.owners = fct_recode(Estimated.owners,
    "0"           = "0 - 0",
    "0-20K"       = "0 - 20000",
    "20K-50K"     = "20000 - 50000",
    "50K-100K"    = "50000 - 100000",
    "100K-200K"   = "100000 - 200000",
    "200K-500K"   = "200000 - 500000",
    "500K-1M"     = "500000 - 1000000",
    "1M-2M"       = "1000000 - 2000000",
    "2M-5M"       = "2000000 - 5000000",
    "5M-10M"      = "5000000 - 10000000",
    "10M-20M"     = "10000000 - 20000000",
    "20M-50M"     = "20000000 - 50000000",
    "50M-100M"    = "50000000 - 100000000",
    "100M-200M"   = "100000000 - 200000000"
  ))

In [ ]:
ggplot(data_rename, aes(x=Estimated.owners)) + 
  geom_bar()+labs(title = "Répartition des ventes (Owners)", x = "Nombre de possesseurs", y = "Nombre de jeux")

On va devoir ordonner les modalités car elle ne permettent pas d'être lu dans l'odre croissant des détenteur des jeux

In [ ]:
ordre_logique <- c(
  "0", "0-20K", "20K-50K", "50K-100K", "100K-200K", 
  "200K-500K", "500K-1M", "1M-2M", "2M-5M", "5M-10M", 
  "10M-20M", "20M-50M", "50M-100M", "100M-200M"
)

data_rename <- data_rename %>%
  mutate(Estimated.owners = fct_relevel(Estimated.owners, ordre_logique))


In [ ]:
ggplot(data_rename, aes(x=Estimated.owners)) + 
  geom_bar()+labs(title = "Répartition des ventes (Owners)", x = "Nombre de possesseurs", y = "Nombre de jeux")

On remarque qu'il y a tres peu de jeux qui sont téléchargés en très grande quantités. La plupart des jeux on entre 0 et 20K utilisateurs.

In [ ]:
nrow(data_rename)

## Etude du prix sur estimated owners

On peut commencer par étudier l'inflence des prix des jeux sur le succés des jeux. Par exemple, qu'elle est la part des jeux gratuits. Pour cela on va créer un nouveau data_frame  sans les jeux gratuit et un DF qui répertorie le nombre de jeux gratuit par tranche et le pourcentage des jeux gratuits par rapport à la totalité des jeux.

In [ ]:

df_percent_free <- data_rename %>%
  group_by(Estimated.owners) %>%
  summarise(
    total = n(),
    gratuits = sum(Price == 0, na.rm = TRUE),
    percent_free = (gratuits / total) * 100
  )


df_wo_free <- data_rename %>%
  filter(Price != 0)

In [ ]:
tail(df_percent_free)
head(df_wo_free)

In [ ]:
ggplot(df_percent_free, aes(x = Estimated.owners, y = percent_free)) +
  geom_col(fill = "darkgreen") +
  coord_flip() +
  theme_minimal() +
  labs(
    title = "Part des jeux gratuits par tranche de possesseurs",
    x = "Tranche d'owners",
    y = "% de jeux gratuits"
  )

On constate que la plupart des jeux gratuits sont les jeux qui "marchent" le plus en pourcentage.

## Inflence variable L_positive et L_negative

Plutôt que d'utiliser les variables metacritic.score ou user.score qui contiennent enormément de 0 (équivalent à pas de note on va analyser les variables transformée LNegative et LPositive.

In [ ]:
ggplot(data_rename, aes(x = Estimated.owners, y = LPositive, fill = Estimated.owners)) +
  geom_boxplot() +
  coord_flip() +
  theme_minimal() +
  labs(title = "Nombre de Reviews Positives (Log) par tranche de succès")

In [ ]:
ggplot(data_rename, aes(x = Estimated.owners, y = LNegative, fill = Estimated.owners)) +
  geom_boxplot() +
  coord_flip() +
  theme_minimal() +
  labs(title = "Nombre de Reviews Negative (Log) par tranche de détention")

L'analyse des boxplots montre une progression quasi linéaire entre le logarithme des reviews (positives et négatives) et les tranches de détention. Nous pouvons en conclure que le succès commercial (Estimated.owners) est l'un des principals facteur du volume d'expression des joueurs. Bien que les deux courbes ont la même tendance, l'échelle atteinte par les critiques positives est nettement supérieure que les critiques négatives. Avec un plafond à environ $e^{15}$  (environ 3,2 millions) contre $e^{13}$ (~440 000) pour les négatives, on constate que le succès sur Steam reste fortement lié à une satisfaction globale. Plus on augmente en popularité plus le nombre de critique tout court mais aussi positive augmente.

Pour vérifier que les tranches de owners les plus élévés est des critiques positives en plus grand nombre que les critiques négatives on va réaliser une ratio (score) de satisfaction. Pour cela je vais revenir au variables sans le log sinon ça va fausser le résultat.

In [ ]:
df_ratio <- data_rename %>%
  mutate(
    Vraies_Positives = exp(LPositive) - 1, 
    Vraies_Negatives = exp(LNegative) - 1,
    
    Total_Reviews = Vraies_Positives + Vraies_Negatives,
    
    # Calcul du vrai ratio de satisfaction
    Satisfaction_Ratio = ifelse(Total_Reviews > 0, 
                                (Vraies_Positives / Total_Reviews) * 100, 
                                NA)
  ) %>%
  group_by(Estimated.owners) %>%
  summarise(
    Moyenne_Satisfaction = mean(Satisfaction_Ratio, na.rm = TRUE),
    Mediane_Satisfaction = median(Satisfaction_Ratio, na.rm = TRUE),
    Nb_Jeux = n()
  )

print(df_ratio)

In [ ]:
ggplot(df_ratio, aes(x = Estimated.owners, y = Moyenne_Satisfaction)) +

  geom_col(fill = "darkolivegreen", alpha = 0.8) +
  theme_minimal() +
  labs(
    title = "Taux de satisfaction moyen par tranche de ventes",
    subtitle = "Moyenne du ratio (Positives / Total) par catégorie d'owners",
    x = "Tranches de owners",
    y = "Satisfaction moyenne (%)"
  )

L'analyse du taux de satisfaction moyen révèle une dynamique non-linéaire. Si la satisfaction croît mécaniquement avec le volume de ventes jusqu'au palier des 5 millions d'owners (validant l'idée que la qualité perçue booste les ventes), on observe une baisse du ratio pour les jeux dépassant les 100 millions de détenteurs. On peut trouver plusieurs raions à cette décroissance commme: 
- les joueurs postent souvent des reviews négatives après beaucoup d'heures de jeu suite à une mise à jour qu'ils n'ont pas aimé, même s'ils aiment le jeu.
- Ou bien quand un jeu devient un vraiment un succes international, il attire des joueurs qui ne sont pas forcément la cible de base, ce qui augmente les critiques négatives.

## Inflence du genre

## Influence du logiciel (windows , mac, linux)

# ACP 

## Création dataframe avec variable quantitative

In [ ]:
df_pca <- data_rename %>%
  select(where(is.numeric)) %>%
  select(-contains("AppID"), -any_of(c("Release.date", "Required.age","Peak.CCU","Metacritic.score","User.score","Achievements","Median.playtime.forever","Average.playtime.forever"))) 


# Vérification de la taille finale
print(paste("Nombre de lignes pour l'ACP :", nrow(df_pca)))
print(colnames(df_pca))

In [ ]:
res.pca <- PCA(df_pca, scale.unit = TRUE, graph = FALSE)

In [ ]:
fviz_eig(res.pca)
res.pca$eig

On regarde quand il y a un coude présent soir garder 4 dimensions cela explique 65% de la variance: c'est un peu bas (inférieur à 80%).On va continuer notre analyse pour voir si on peu en dire plus de choses.

In [ ]:
fviz_pca_var(res.pca,axes = c(1,2),repel=TRUE)
fviz_pca_var(res.pca,axes = c(1,3),repel=TRUE)
fviz_pca_var(res.pca,axes = c(1,4),repel=TRUE)
fviz_pca_var(res.pca,axes = c(2,3),repel=TRUE)
fviz_pca_var(res.pca,axes = c(2,4),repel=TRUE)


Commentaires:

In [ ]:
fviz_pca_biplot(res.pca,axes = c(1,2),repel=TRUE)
fviz_pca_biplot(res.pca,axes = c(1,3),repel=TRUE)
fviz_pca_biplot(res.pca,axes = c(1,4),repel=TRUE)
fviz_pca_biplot(res.pca,axes = c(2,3),repel=TRUE)
fviz_pca_biplot(res.pca,axes = c(2,4),repel=TRUE)


In [ ]:
fviz_pca_biplot(res.pca,axes = c(2,4))